# DDPM on MNIST

Here, we want to implement and train a simple diffusion model on the MNIST data set, which learns to generate images of handwritten digits.

### Imports, loading the data and fundamental classes


In [2]:
from torch_lr_finder import LRFinder
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
from torch.utils.data import Subset

batch_size = 32

import numpy as np
from tqdm.notebook import tqdm


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

/home/scur0036/.conda/envs/diffusion/lib/python3.10/site-packages/torch_lr_finder/lr_finder.py:5: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


cpu


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # necessary so cuda only sees the one available device

In [4]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.version.cuda)

False
0
13.0


In [5]:
class NoiseScheduler(nn.Module):
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        betas = torch.linspace(beta_start, beta_end, T)
        alphas = 1 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)

        # register as buffers — not parameters, but move with .to(device)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_bars', alpha_bars)
        self.T = T

    def _reshape(self, x, t):
        return x.reshape(-1, 1, 1, 1) if isinstance(t, (torch.Tensor, np.ndarray)) else x

    def beta(self, t): return self._reshape(self.betas[t], t)
    def alpha(self, t): return self._reshape(self.alphas[t], t)
    def alpha_bar(self, t): return self._reshape(self.alpha_bars[t], t)

    def add_noise(self, x, t):
        noise = torch.randn_like(x)
        ab = self.alpha_bar(t)
        return torch.sqrt(ab) * x + torch.sqrt(1 - ab) * noise, noise


scheduler = NoiseScheduler(T=1000)

NoisyMNIST is the data set on which the MNIST is trained: (t,noisy image) -> noise pattern

In [6]:
# cell written by Claude
class NoisyMNIST(torch.utils.data.Dataset):
    def __init__(self, dataset, scheduler):
        self.dataset = dataset
        self.scheduler = scheduler

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # the device on which the scheduler lies
        scheduler_device = self.scheduler.betas.device
        # the second entry (the number) is being ignored
        x, _ = self.dataset[idx]
        t = torch.randint(0, self.scheduler.T, (1,))
        x_noisy, noise = self.scheduler.add_noise(x.unsqueeze(0).to(scheduler_device), t.to(
            scheduler_device))  # TODO: maybe i rather send the scheduler to the device of the dataset
        return x_noisy.squeeze(0),t,noise.squeeze(0)


In [7]:
# cell generated by Claude
def zeros_only(dataset):
    indices = [i for i, (_, label) in enumerate(dataset) if label == 0]
    return Subset(dataset, indices)

def load_mnist(transform=None):
    if transform is None:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
    train_set = datasets.MNIST(
        root='./data', train=True,  download=True, transform=transform)
    test_set = datasets.MNIST(
        root='./data', train=False, download=True, transform=transform)
    return train_set, test_set


def get_zeros_loaders(train_set, test_set, scheduler, batch_size=32):
    train_zeros = NoisyMNIST(zeros_only(train_set), scheduler)
    test_zeros = NoisyMNIST(zeros_only(test_set), scheduler)
    train_loader = DataLoader(train_zeros, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_zeros,  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

In [8]:
train_set, test_set = load_mnist()
train_loader, test_loader = get_zeros_loaders(
    train_set, test_set, scheduler, batch_size)

## Part 1: Building the initial unet
We are editing this architecture now to include time embeddings. Look at this article: https://medium.com/@ikim1994914/fundamental-generative-ai-part-2b-u-net-with-self-attention-and-time-embedding-057377f503ab

We will build a time embedder. We start with a linear embedder, but later move on to sinusoidal time embedding (which is the standard or at least more common). Every pixel will then receive a time embedding associated with it




In [11]:
class LinearTimeEmbedding(nn.Module):
    def __init__(self, d=32, hidden_dim=128):
        super().__init__()
        self.d = d
        self.proj = nn.Sequential(
            nn.Linear(d, hidden_dim),
            nn.SiLU(),  # SiLU (swish) is standard in diffusion models, slightly better than ReLU here
        )
    
    def forward(self, t):
        # t: (B,) -> (B, d)
        emb = t.unsqueeze(1).expand(-1, self.d).float()  # linear: just repeat t
        return self.proj(emb)  # (B, hidden_dim)

In [10]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, d=32, hidden_dim=128):
        super().__init__()
        self.d = d
        self.proj = nn.Sequential(
            nn.Linear(d, hidden_dim),
            nn.SiLU(),
        )
    
    def forward(self, t):
        # t: (B,)
        device = t.device
        half_dim = self.d // 2
        
        # 1. Frequenzen vorbereiten (log-space für bessere numerische Stabilität)
        # exponent reicht von 0 bis 1
        emb = torch.log(10000) / (half_dim - 1) 
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb) # = 10000^(-(0,1,2,...,d/2)/(d/2-1) this is not exactly the sinusoidal embedding formula when written down, but it leads to a better distribution of values while maintaining the same purpose kindof
        
        # 2. t mit Frequenzen multiplizieren
        # (B, 1) * (half_dim,) -> (B, half_dim)
        emb = t[:, None] * emb[None, :]
        
        # 3. Sinus und Kosinus kombinieren
        # (B, half_dim * 2) -> (B, d)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        
        # 4. Durch das MLP jagen
        return self.proj(emb) # (B, hidden_dim)


TODO: make it possible to set the time embedding from the outside

In [ ]:
from typing import List


class UNet(nn.Module):

    def __init__(self, channels: List[int] = [32, 64, 128], convs_per_level=2,
                 kernel_size=3, pool_size=2, padding=1,
                 d_time_emb = 32,hidden_dim_time_emb = 128,embedding = "Linear"):
        super().__init__()

        if embedding=="linear":
            self.time_emb = LinearTimeEmbedding(d_time_emb,hidden_dim_time_emb)
        elif embedding == "sinusoidal":
            self.time_emb = SinusoidalTimeEmbedding(d_time_emb,hidden_dim_time_emb)
        else:
            raise Exception("Allowed inputs for embedding are 'linear' or 'sinusoidal'")
        
        full_channels = [1] + channels + channels[-2::-1]
        # [1, 8, 16, 32, 16, 8] 
        
        self.mid = len(full_channels) // 2

        self.encoder_convs = nn.ModuleList([
            # one ModuleList per resolution level (so this is a Module List of BLocks of N convs)
            # first conv changes channels: full_channels[i] -> full_channels[i+1]
            # remaining convs keep channels constant: full_channels[i+1] -> full_channels[i+1]
            # example with channels=[8,16,32], convs_per_level=2:
            #   level 0 (28x28): Conv(1->8),  Conv(8->8)
            #   level 1 (14x14): Conv(8->16), Conv(16->16)
            #   level 2 (7x7):   Conv(16->32), Conv(32->32)  <- bottleneck, no pool after
            nn.ModuleList([
                nn.Conv2d(full_channels[i] if j == 0 else full_channels[i+1],
                          full_channels[i+1], kernel_size, padding=padding)
                for j in range(convs_per_level)
            ])
            for i in range(self.mid)
        ])

        self.decoder_convs = nn.ModuleList([
            # one ModuleList per resolution level
            # first conv takes concatenated skip+upsampled: (full_channels[mid+i] + full_channels[mid-i-1]) -> full_channels[mid+i+1]
            # remaining convs keep channels constant: full_channels[mid+i+1] -> full_channels[mid+i+1]
            # example with channels=[8,16,32], convs_per_level=2:
            #   level 0 (7x7->14x14):  Conv(32+16->16), Conv(16->16)
            #   level 1 (14x14->28x28): Conv(16+8->8),  Conv(8->8)
            # then output_conv: Conv(8->1, kernel=1)
            nn.ModuleList([
                nn.Conv2d(
                    (full_channels[self.mid + i] + full_channels[self.mid - i - 1]) if j == 0
                    else full_channels[self.mid + i + 1],
                    full_channels[self.mid + i + 1],
                    kernel_size, padding=padding
                )
                for j in range(convs_per_level)
            ])
            for i in range(len(full_channels) - self.mid - 1)
        ])

        # final output conv: 8 -> 1, no activation
        self.output_conv = nn.Conv2d(full_channels[-1], 1, kernel_size=1)

        self.pool = nn.MaxPool2d(pool_size)
        self.upsample = nn.Upsample(scale_factor=pool_size)
        self.activation_fn = F.relu

    def forward(self, x):
        skips = []
        last_encoder = len(self.encoder_convs) - 1
        for i, conv_block in enumerate(self.encoder_convs):
            for conv in conv_block:
                x = self.activation_fn(conv(x))
            if i < last_encoder:
                skips.append(x)
                x = self.pool(x)

        for i, conv_block in enumerate(self.decoder_convs):
            x = self.upsample(x)
            x = torch.cat([x, skips.pop()], dim=1)
            for conv in conv_block:
                x = self.activation_fn(conv(x))

        return self.output_conv(x)  # no activation on final layer

### training the model


For speeding up, i use autocast and GradScaler, which, where it is possible, scales numbers to half precision (16 bit), which on gpus allows for significant improvements in speed. GradScaler and autocast deal with issues that can arise here when we just use .half().


In [10]:
def find_lr(model, train_loader, start_lr=1e-7, end_lr=1, num_iter=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=start_lr)
    lr_finder = LRFinder(model, optimizer, F.mse_loss)
    lr_finder.range_test(train_loader, end_lr=end_lr, num_iter=num_iter)
    lr_finder.plot()
    lr_finder.reset()
    return lr_finder

In [11]:
def train(model, train_loader, test_loader, epochs=100, lr=1e-2, plot_loss=True,
          early_stopping_patience=10, save_path='model.pkl'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda')
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5, min_lr=1e-6)

    loss_fn = nn.MSELoss()
    train_losses, test_losses = [], []

    best_test_loss = float('inf')
    patience_counter = 0

    for epoch in tqdm(range(epochs), desc='Epochs'):
        model.train()
        epoch_train_loss = 0
        for x_noisy, noise in tqdm(train_loader, leave=False, desc='train'):
            x_noisy, noise = x_noisy.to(device), noise.to(device)
            with torch.amp.autocast('cuda'):
                noise_pred = model(x_noisy)
                loss = loss_fn(noise_pred, noise)
            epoch_train_loss += loss.item()
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        train_losses.append(epoch_train_loss / len(train_loader))

        model.eval()
        epoch_test_loss = 0
        with torch.no_grad():
            for x_noisy, noise in tqdm(test_loader, leave=False, desc='test'):
                x_noisy, noise = x_noisy.to(device), noise.to(device)
                with torch.amp.autocast('cuda'):
                    noise_pred = model(x_noisy)
                    epoch_test_loss += loss_fn(noise_pred, noise).item()
            test_loss = epoch_test_loss / len(test_loader)
            lr_scheduler.step(test_loss)
        test_losses.append(test_loss)

        print(
            f"Epoch {epoch} | train loss: {train_losses[-1]:.4f} | test loss: {test_losses[-1]:.4f}")

        # early stopping
        if test_losses[-1] < best_test_loss:
            best_test_loss = test_losses[-1]
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # restore best model
    model.load_state_dict(torch.load(save_path))

    if plot_loss:
        plt.figure(figsize=(8, 4))
        plt.plot(train_losses, label='Train')
        plt.plot(test_losses, label='Test')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Loss curve')
        plt.legend()
        plt.tight_layout()
        plt.show()

    return train_losses, test_losses

In [17]:
channels = [64, 128, 256]
model = UNet(channels,convs_per_level=1).to(device)
find_lr(model, train_loader)

  0%|          | 0/100 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (28) must match the size of tensor b (32) at non-singleton dimension 2

In [ ]:
model = UNet(channels).to(device)

lr = 6e-3

train(model, train_loader, test_loader,
      epochs=15, lr=lr, early_stopping_patience=15,
      save_path='model_unet_no_t_emb.pkl')

### Generating an image


In [85]:
example_image = train_set[0][0].squeeze()


def noisy_image(n=1):
    """generates a batch of n noisy images. Output shape: (nx28x28)"""
    return torch.randn((n, 1, *example_image.shape))


plt.imshow(noisy_image().squeeze(), cmap="gray")
plt.show()

In [96]:
def generate_image(unet, scheduler, stochasticity=1.0, n_images=1, return_intermediates=False):
    intermediates = []
    x = noisy_image(n_images).to(device)
    if return_intermediates:
        intermediates.append(x)
    for t in range(scheduler.T-1, 0, -1):
        z = noisy_image(n_images).to(device) if t > 1 else torch.zeros_like(x)
        alpha = scheduler.alpha(t)
        alpha_bar = scheduler.alpha_bar(t)
        # sigma_t is here for this simple case set to sqrt(beta), like in the original ho et al paper
        sigma_t = stochasticity*torch.sqrt(scheduler.beta(t))

        noise_pred = unet(x)
        x = 1/torch.sqrt(alpha) * (x - (1-alpha) /
                                   torch.sqrt(1-alpha_bar)*noise_pred) + sigma_t*z

        # print(t, alpha_bar.item(), (1-alpha_bar).item())

        if return_intermediates:
            intermediates.append(x)

    if return_intermediates:
        return x, intermediates

    return x

In [87]:
def plot_image(image):
    plt.imshow(image.cpu().detach().squeeze(), cmap="gray")
    plt.colorbar()

In [99]:
scheduler = NoiseScheduler(T=1000)

image, intermediates = generate_image(model, scheduler, stochasticity=1,
                                      n_images=1, return_intermediates=True)
plot_image(image)

### changing stochasticity (i.e. DDPM vs DDIM)

we can change how much noise we add back into the image


In [103]:
def plot_stochasticities(model, scheduler, stochasticities=[0, 0.2, 0.4, 0.6, 0.8, 1.0], ncols=3):
    nrows = len(stochasticities) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*2, nrows*2))
    
    for ax, s in zip(axes.flat, stochasticities):
        image = generate_image(model, scheduler, stochasticity=s, n_images=1)
        ax.imshow(image.cpu().detach().squeeze(), cmap='gray')
        ax.set_title(f's={s}')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_stochasticities(model,scheduler)

### implementing the t embedding


Steps:

time embedding

use cosine based variance schedule (nochol & dhariwal)

add attention layer to the unet

conditioning on input numbers
